# 03 Benchmark Species

Benchmark des architectures pour la classification d'espece.

Execution volontairement manuelle: chaque architecture a sa propre cellule. Tu peux en lancer une, interrompre, redemarrer le kernel, puis continuer plus tard sans perdre le comparatif deja sauvegarde.

## DagsHub / MLflow

Avant de lancer les cellules d'entrainement:
- creer le repo DagsHub lie au repo GitHub si ce n'est pas deja fait,
- verifier `.env` avec `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`,
- lancer le notebook en local depuis l'environnement GPU.

Tu n'as pas besoin de pousser sur GitHub avant chaque run: MLflow logge directement depuis ta machine vers DagsHub. Pour les gros runs de benchmark, pousse quand meme ton code avant de commencer afin de pouvoir relier les resultats a une version claire du projet.

In [26]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import tensorflow as tf

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core.mlflow_config import setup_mlflow
from src.data.augmentation import IMAGE_SIZE
from src.models.build import (
    build_model,
    densenet_preprocess_input,
    list_available_architectures,
    list_recommended_screening_architectures,
    resnet_v2_preprocess_input,
)
from src.models.train import TrainingConfig, train_model

tracking_uri = setup_mlflow(experiment_name="species-benchmark")
tracking_uri


2026-04-11 18:09:50.757 | INFO     | src.core.mlflow_config:setup_mlflow:39 - MLflow configure vers https://dagshub.com/thomashebert99/Plant-disease-detection.mlflow


'https://dagshub.com/thomashebert99/Plant-disease-detection.mlflow'

In [27]:
SEED = 42
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 8
PREFETCH_BUFFER_SIZE = 1
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
DATA_ROOT = REPO_ROOT / "data" / "processed" / "species"
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"
OOD_ROOT = REPO_ROOT / "data" / "test_ood"
MODEL_ROOT = REPO_ROOT / "models" / "species"
HISTORY_ROOT = MODEL_ROOT / "histories"
RESULTS_PATH = MODEL_ROOT / "benchmark_results.csv"

if not TRAIN_DIR.exists() or not VAL_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError(
        f"Dossiers attendus introuvables: {TRAIN_DIR}, {VAL_DIR}, {TEST_DIR}. "
        "Relance l'etape 4 avec le split train/val/test avant ce benchmark."
    )

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
HISTORY_ROOT.mkdir(parents=True, exist_ok=True)

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="int",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    label_mode="int",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode="int",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
AUTOTUNE = tf.data.AUTOTUNE

train_labels = []
for class_index, class_name in enumerate(class_names):
    class_dir = TRAIN_DIR / class_name
    image_count = sum(
        1
        for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    train_labels.extend([class_index] * image_count)

class_indices = np.arange(NUM_CLASSES)
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=class_indices,
    y=np.array(train_labels),
)
class_weight = {
    int(class_index): float(weight)
    for class_index, weight in zip(class_indices, class_weight_values)
}
class_weight_df = pd.DataFrame(
    {
        "class_index": class_indices,
        "class_name": class_names,
        "class_weight": class_weight_values,
    }
)


def load_raw_image(image_path: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image, label


def build_ood_species_dataset() -> tuple[tf.data.Dataset | None, pd.DataFrame]:
    records = []
    if not OOD_ROOT.exists():
        return None, pd.DataFrame()

    for class_index, class_name in enumerate(class_names):
        species_dir = OOD_ROOT / class_name
        if not species_dir.exists():
            continue
        for image_path in sorted(species_dir.rglob("*")):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(
                    {
                        "image_path": str(image_path),
                        "class_index": class_index,
                        "class_name": class_name,
                    }
                )

    frame = pd.DataFrame(records)
    if frame.empty:
        return None, frame

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            frame["image_path"].to_numpy(),
            frame["class_index"].to_numpy(dtype="int32"),
        )
    )
    dataset = dataset.map(load_raw_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(EVAL_BATCH_SIZE).prefetch(PREFETCH_BUFFER_SIZE)
    return dataset, frame


train_ds = train_ds.prefetch(PREFETCH_BUFFER_SIZE)
val_ds = val_ds.prefetch(PREFETCH_BUFFER_SIZE)
test_ds = test_ds.prefetch(PREFETCH_BUFFER_SIZE)
ood_species_ds, ood_species_df = build_ood_species_dataset()

class_weight_df, ood_species_df.groupby("class_name").size().rename("ood_count")


Found 47125 files belonging to 7 classes.
Found 10101 files belonging to 7 classes.
Found 10092 files belonging to 7 classes.


(   class_index  class_name  class_weight
 0            0       apple      0.990021
 1            1        corn      1.051569
 2            2       grape      1.065550
 3            3      pepper      1.971922
 4            4      potato      1.349127
 5            5  strawberry      2.137867
 6            6      tomato      0.419396,
 class_name
 apple         287
 corn          378
 grape         154
 pepper        125
 potato        379
 strawberry     96
 tomato        903
 Name: ood_count, dtype: int64)

In [29]:
SCREENING_ARCHITECTURES = list_recommended_screening_architectures()
DEFERRED_ARCHITECTURES = [
    "EfficientNetB3",
    "ResNet50V2",
    "ResNet101V2",
    "ConvNeXtSmall",
    "DenseNet169",
]
ARCHITECTURE_FAMILY = {
    "MobileNetV3Small": "MobileNet",
    "MobileNetV3Large": "MobileNet",
    "EfficientNetB0": "EfficientNet",
    "EfficientNetB1": "EfficientNet",
    "EfficientNetB3": "EfficientNet",
    "ResNet50V2": "ResNet",
    "ResNet101V2": "ResNet",
    "ConvNeXtTiny": "ConvNeXt",
    "ConvNeXtSmall": "ConvNeXt",
    "DenseNet121": "DenseNet",
    "DenseNet169": "DenseNet",
}
SCREENING_RANKING_METRICS = [
    ("eval_val_f1_macro", 0.30, True),
    ("eval_val_balanced_accuracy", 0.20, True),
    ("eval_val_log_loss", 0.20, False),
    ("eval_val_ms_per_image", 0.15, False),
    ("loss_gap", 0.10, False),
    ("overfit_gap", 0.05, False),
]
FINAL_RANKING_METRICS = [
    ("eval_test_f1_macro", 0.25, True),
    ("eval_test_balanced_accuracy", 0.20, True),
    ("eval_ood_f1_macro", 0.20, True),
    ("eval_ood_balanced_accuracy", 0.15, True),
    ("eval_test_log_loss", 0.10, False),
    ("eval_test_ms_per_image", 0.05, False),
    ("loss_gap", 0.05, False),
]
DEFAULT_FINE_TUNE_LAYERS = [20, 50]
DEFAULT_FINE_TUNE_LR = [1e-5, 3e-5]
PREPROCESSOR_CUSTOM_OBJECTS = {
    "DenseNet121": {
        "preprocess_input": tf.keras.applications.densenet.preprocess_input,
        "densenet_preprocess_input": densenet_preprocess_input,
        "plant_disease>densenet_preprocess_input": densenet_preprocess_input,
    },
    "DenseNet169": {
        "preprocess_input": tf.keras.applications.densenet.preprocess_input,
        "densenet_preprocess_input": densenet_preprocess_input,
        "plant_disease>densenet_preprocess_input": densenet_preprocess_input,
    },
    "ResNet50V2": {
        "preprocess_input": tf.keras.applications.resnet_v2.preprocess_input,
        "resnet_v2_preprocess_input": resnet_v2_preprocess_input,
        "plant_disease>resnet_v2_preprocess_input": resnet_v2_preprocess_input,
    },
    "ResNet101V2": {
        "preprocess_input": tf.keras.applications.resnet_v2.preprocess_input,
        "resnet_v2_preprocess_input": resnet_v2_preprocess_input,
        "plant_disease>resnet_v2_preprocess_input": resnet_v2_preprocess_input,
    },
}
SCREENING_RESULTS_PATH = MODEL_ROOT / "screening_results.csv"
FINE_TUNE_RESULTS_PATH = MODEL_ROOT / "finetune_results.csv"

missing_architectures = sorted(
    set(SCREENING_ARCHITECTURES + DEFERRED_ARCHITECTURES) - set(list_available_architectures())
)
if missing_architectures:
    raise ValueError(f"Architectures indisponibles: {missing_architectures}")

benchmark_results = {}
trained_models = {}
evaluation_details = {}


def safe_run_name(value: str) -> str:
    return value.strip().replace("/", "_").replace(" ", "_").replace(".", "p").replace("-", "_")


def sort_summary(summary: pd.DataFrame) -> pd.DataFrame:
    for column in ["eval_test_f1_macro", "eval_ood_f1_macro", "eval_val_f1_macro"]:
        if column in summary.columns:
            return summary.sort_values(column, ascending=False)
    return summary


def load_summary(results_path: Path = FINE_TUNE_RESULTS_PATH) -> pd.DataFrame:
    if results_path.exists():
        return sort_summary(pd.read_csv(results_path))
    return pd.DataFrame()


def save_result(row: dict[str, object], results_path: Path) -> pd.DataFrame:
    summary = load_summary(results_path)
    if not summary.empty and "run_name" in summary.columns:
        summary = summary[summary["run_name"] != row["run_name"]]
    summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True)
    summary = sort_summary(summary)
    summary.to_csv(results_path, index=False)
    return summary


def save_history(run_name: str, result: dict[str, object]) -> Path:
    history_path = HISTORY_ROOT / f"{run_name}_history.json"
    payload = {
        "phase1": result["phase1_history"],
        "phase2": result["phase2_history"],
    }
    history_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return history_path


def load_history(run_name: str) -> dict[str, dict[str, list[float]]]:
    history_path = HISTORY_ROOT / f"{run_name}_history.json"
    if not history_path.exists():
        raise FileNotFoundError(
            f"Historique introuvable pour {run_name}. "
            "Lance d'abord la cellule d'entrainement correspondante."
        )
    return json.loads(history_path.read_text(encoding="utf-8"))


def predict_species_dataset(
    model: tf.keras.Model,
    dataset: tf.data.Dataset,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, int, float]:
    y_true = []
    y_score_batches = []
    image_count = 0
    started_at = time.perf_counter()

    for images, labels in dataset:
        predictions = model.predict(images, verbose=0)
        y_score_batches.append(predictions)
        y_true.extend(labels.numpy().astype(int).tolist())
        image_count += int(images.shape[0])

    elapsed = time.perf_counter() - started_at
    y_score = np.concatenate(y_score_batches, axis=0)
    y_true = np.array(y_true, dtype="int32")
    y_pred = np.argmax(y_score, axis=1)
    return y_true, y_pred, y_score, image_count, elapsed


def evaluate_species_model(
    run_name: str,
    architecture: str,
    model: tf.keras.Model,
    dataset: tf.data.Dataset,
    eval_name: str,
) -> dict[str, float]:
    y_true, y_pred, y_score, image_count, elapsed = predict_species_dataset(model, dataset)
    labels = np.arange(NUM_CLASSES)
    accuracy = float(accuracy_score(y_true, y_pred))
    top_k = min(2, NUM_CLASSES - 1) if NUM_CLASSES > 2 else 1

    metrics = {
        f"eval_{eval_name}_accuracy": accuracy,
        f"eval_{eval_name}_error_rate": float(1.0 - accuracy),
        f"eval_{eval_name}_balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        f"eval_{eval_name}_precision_macro": float(
            precision_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        f"eval_{eval_name}_recall_macro": float(
            recall_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        f"eval_{eval_name}_f1_macro": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        f"eval_{eval_name}_log_loss": float(log_loss(y_true, y_score, labels=labels)),
        f"eval_{eval_name}_ms_per_image": float((elapsed / image_count) * 1000),
    }
    if top_k > 1:
        metrics[f"eval_{eval_name}_top{top_k}_accuracy"] = float(
            top_k_accuracy_score(y_true, y_score, k=top_k, labels=labels)
        )

    evaluation_details[(run_name, eval_name)] = {
        "y_true": y_true,
        "y_pred": y_pred,
        "y_score": y_score,
        "metrics": metrics,
    }

    try:
        with mlflow.start_run(run_name=f"{run_name}_eval_{eval_name}"):
            mlflow.log_params(
                {
                    "architecture": architecture,
                    "run_name": run_name,
                    "task": "species",
                    "phase": f"eval_{eval_name}",
                }
            )
            mlflow.log_metrics(metrics)
    except Exception as exc:
        print(
            f"Warning MLflow: logging evaluation {run_name}/{eval_name} ignore: {exc}"
        )

    return metrics


def best_metric(history: dict[str, list[float]], name: str) -> float | None:
    values = history.get(name, [])
    if not values:
        return None
    return float(min(values) if "loss" in name else max(values))


def overfit_gap(history: dict[str, list[float]]) -> float | None:
    accuracy = history.get("accuracy", [])
    val_accuracy = history.get("val_accuracy", [])
    if not accuracy or not val_accuracy:
        return None
    return float(accuracy[-1] - val_accuracy[-1])


def loss_gap(history: dict[str, list[float]]) -> float | None:
    loss = history.get("loss", [])
    val_loss = history.get("val_loss", [])
    if not loss or not val_loss:
        return None
    return float(val_loss[-1] - loss[-1])


def diagnose_overfitting(history: dict[str, list[float]]) -> str:
    accuracy_gap = overfit_gap(history)
    validation_loss_gap = loss_gap(history)
    val_loss = history.get("val_loss", [])

    if accuracy_gap is None or validation_loss_gap is None or len(val_loss) < 2:
        return "diagnostic_indisponible"

    val_loss_worsens = val_loss[-1] > min(val_loss[:-1]) * 1.05
    if accuracy_gap > 0.05 and validation_loss_gap > 0.20 and val_loss_worsens:
        return "overfitting_probable"
    if accuracy_gap > 0.03 or validation_loss_gap > 0.10:
        return "a_surveilleiller"
    return "pas_d_overfitting_significatif"


def build_result_row(
    *,
    run_name: str,
    architecture: str,
    stage: str,
    result: dict[str, object],
    history_path: Path,
    phase1_learning_rate: float,
    phase2_learning_rate: float | None,
    fine_tune_layers: int,
    eval_metrics: dict[str, float],
) -> dict[str, object]:
    phase1_history = result["phase1_history"]
    phase2_history = result["phase2_history"]
    diagnostic_history = phase2_history or phase1_history

    return {
        "run_name": run_name,
        "architecture": architecture,
        "architecture_family": ARCHITECTURE_FAMILY.get(architecture, "unknown"),
        "stage": stage,
        "phase1_learning_rate": phase1_learning_rate,
        "phase2_learning_rate": phase2_learning_rate,
        "fine_tuned_layers": result["fine_tuned_layers"],
        "requested_fine_tune_layers": fine_tune_layers,
        "phase1_best_val_accuracy": best_metric(phase1_history, "val_accuracy"),
        "phase1_best_val_loss": best_metric(phase1_history, "val_loss"),
        "phase2_best_val_accuracy": best_metric(phase2_history, "val_accuracy"),
        "phase2_best_val_loss": best_metric(phase2_history, "val_loss"),
        "overfit_gap": overfit_gap(diagnostic_history),
        "loss_gap": loss_gap(diagnostic_history),
        "overfit_diagnostic": diagnose_overfitting(diagnostic_history),
        "checkpoint_path": str(result["checkpoint_path"]),
        "history_path": str(history_path),
        "class_weight": str(class_weight),
        **eval_metrics,
    }


def load_species_checkpoint(architecture: str, checkpoint_path: Path) -> tf.keras.Model:
    return tf.keras.models.load_model(
        str(checkpoint_path),
        compile=False,
        safe_mode=False,
        custom_objects=PREPROCESSOR_CUSTOM_OBJECTS.get(architecture, {}),
    )


def evaluate_checkpoint(
    *,
    run_name: str,
    architecture: str,
    checkpoint_path: Path,
    include_test: bool,
    include_ood: bool,
    keep_model: bool,
) -> dict[str, float]:
    tf.keras.backend.clear_session()
    eval_model = load_species_checkpoint(architecture, checkpoint_path)

    eval_metrics = evaluate_species_model(run_name, architecture, eval_model, val_ds, "val")
    if include_test:
        eval_metrics.update(
            evaluate_species_model(run_name, architecture, eval_model, test_ds, "test")
        )
    if include_ood and ood_species_ds is not None:
        eval_metrics.update(
            evaluate_species_model(run_name, architecture, eval_model, ood_species_ds, "ood")
        )

    if keep_model:
        trained_models[run_name] = eval_model
    else:
        del eval_model
        tf.keras.backend.clear_session()

    return eval_metrics


def run_species_phase1_screening(
    architecture: str,
    *,
    phase1_epochs: int = 4,
    phase1_learning_rate: float = 1e-3,
    include_ood: bool = False,
    keep_model: bool = False,
) -> pd.DataFrame:
    run_name = safe_run_name(f"{architecture}_screening")
    training_model = build_model(architecture=architecture, num_classes=NUM_CLASSES)
    config = TrainingConfig(
        phase1_epochs=phase1_epochs,
        phase2_epochs=0,
        phase1_learning_rate=phase1_learning_rate,
        checkpoint_root=MODEL_ROOT,
    )

    result = train_model(
        model=training_model,
        train_data=train_ds,
        val_data=val_ds,
        architecture=architecture,
        task="species_screening",
        config=config,
        class_weight=class_weight,
        log_to_mlflow=True,
    )
    history_path = save_history(run_name, result)
    del training_model

    eval_metrics = evaluate_checkpoint(
        run_name=run_name,
        architecture=architecture,
        checkpoint_path=result["checkpoint_path"],
        include_test=False,
        include_ood=include_ood,
        keep_model=keep_model,
    )
    row = build_result_row(
        run_name=run_name,
        architecture=architecture,
        stage="screening_phase1",
        result=result,
        history_path=history_path,
        phase1_learning_rate=phase1_learning_rate,
        phase2_learning_rate=None,
        fine_tune_layers=0,
        eval_metrics=eval_metrics,
    )
    return save_result(row, SCREENING_RESULTS_PATH)


def run_species_fine_tuning(
    architecture: str,
    *,
    fine_tune_layers: int = 50,
    phase2_learning_rate: float = 1e-5,
    phase2_epochs: int = 10,
    include_ood: bool = True,
    keep_model: bool = False,
) -> pd.DataFrame:
    lr_name = safe_run_name(f"{phase2_learning_rate:.0e}")
    run_name = safe_run_name(f"{architecture}_ft_l{fine_tune_layers}_lr{lr_name}")
    screening_checkpoint = MODEL_ROOT / f"{architecture}_species_screening" / "best_model.keras"
    if not screening_checkpoint.exists():
        raise FileNotFoundError(
            f"Checkpoint phase 1 introuvable pour {architecture}: {screening_checkpoint}. "
            "Lance d'abord la cellule de screening de ce modele."
        )

    tf.keras.backend.clear_session()
    training_model = load_species_checkpoint(architecture, screening_checkpoint)
    config = TrainingConfig(
        phase1_epochs=0,
        phase2_epochs=phase2_epochs,
        phase2_learning_rate=phase2_learning_rate,
        fine_tune_layers=fine_tune_layers,
        checkpoint_root=MODEL_ROOT,
    )

    result = train_model(
        model=training_model,
        train_data=train_ds,
        val_data=val_ds,
        architecture=architecture,
        task=f"species_ft_l{fine_tune_layers}_lr{lr_name}",
        config=config,
        class_weight=class_weight,
        log_to_mlflow=True,
    )
    history_path = save_history(run_name, result)
    del training_model

    eval_metrics = evaluate_checkpoint(
        run_name=run_name,
        architecture=architecture,
        checkpoint_path=result["checkpoint_path"],
        include_test=True,
        include_ood=include_ood,
        keep_model=keep_model,
    )
    row = build_result_row(
        run_name=run_name,
        architecture=architecture,
        stage="fine_tuning",
        result=result,
        history_path=history_path,
        phase1_learning_rate=1e-3,
        phase2_learning_rate=phase2_learning_rate,
        fine_tune_layers=fine_tune_layers,
        eval_metrics=eval_metrics,
    )
    return save_result(row, FINE_TUNE_RESULTS_PATH)


def recover_fine_tune_from_checkpoint(
    architecture: str,
    *,
    fine_tune_layers: int = 50,
    phase2_learning_rate: float = 1e-5,
    include_ood: bool = True,
    keep_model: bool = False,
) -> pd.DataFrame:
    """Evaluate an already-saved fine-tuned checkpoint after a late MLflow failure."""

    lr_name = safe_run_name(f"{phase2_learning_rate:.0e}")
    run_name = safe_run_name(f"{architecture}_ft_l{fine_tune_layers}_lr{lr_name}")
    checkpoint_path = (
        MODEL_ROOT
        / f"{architecture}_species_ft_l{fine_tune_layers}_lr{lr_name}"
        / "best_model.keras"
    )
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint fine-tune introuvable: {checkpoint_path}")

    result = {
        "checkpoint_path": checkpoint_path,
        "fine_tuned_layers": fine_tune_layers,
        "phase1_history": {},
        "phase2_history": {},
    }
    history_path = HISTORY_ROOT / f"{run_name}_history.json"
    if not history_path.exists():
        history_path = save_history(run_name, result)

    eval_metrics = evaluate_checkpoint(
        run_name=run_name,
        architecture=architecture,
        checkpoint_path=checkpoint_path,
        include_test=True,
        include_ood=include_ood,
        keep_model=keep_model,
    )
    row = build_result_row(
        run_name=run_name,
        architecture=architecture,
        stage="fine_tuning_recovered",
        result=result,
        history_path=history_path,
        phase1_learning_rate=1e-3,
        phase2_learning_rate=phase2_learning_rate,
        fine_tune_layers=fine_tune_layers,
        eval_metrics=eval_metrics,
    )
    row["overfit_diagnostic"] = "historique_indisponible_apres_echec_mlflow"
    return save_result(row, FINE_TUNE_RESULTS_PATH)



def current_finalist_architectures(top_k: int = 4) -> list[str]:
    if "finalist_architectures" in globals() and finalist_architectures:
        return list(finalist_architectures)
    return select_screening_finalists(top_k=top_k)


def get_finalist_architecture(finalist_rank: int, *, top_k: int = 4) -> str:
    finalists = current_finalist_architectures(top_k=top_k)
    if finalist_rank < 1 or finalist_rank > len(finalists):
        raise IndexError(
            f"Finaliste {finalist_rank} indisponible. "
            f"Finalistes actuels: {finalists}"
        )
    return finalists[finalist_rank - 1]


def fine_tune_run_name(
    architecture: str,
    *,
    fine_tune_layers: int = 50,
    phase2_learning_rate: float = 1e-5,
) -> str:
    lr_name = safe_run_name(f"{phase2_learning_rate:.0e}")
    return safe_run_name(f"{architecture}_ft_l{fine_tune_layers}_lr{lr_name}")


def run_finalist_fine_tuning(
    finalist_rank: int,
    *,
    fine_tune_layers: int = 50,
    phase2_learning_rate: float = 1e-5,
    phase2_epochs: int = 10,
    include_ood: bool = True,
    keep_model: bool = False,
) -> pd.DataFrame:
    architecture = get_finalist_architecture(finalist_rank)
    print(f"Fine-tuning finaliste {finalist_rank}: {architecture}")
    return run_species_fine_tuning(
        architecture,
        fine_tune_layers=fine_tune_layers,
        phase2_learning_rate=phase2_learning_rate,
        phase2_epochs=phase2_epochs,
        include_ood=include_ood,
        keep_model=keep_model,
    )


def inspect_finalist_fine_tuning(
    finalist_rank: int,
    *,
    fine_tune_layers: int = 50,
    phase2_learning_rate: float = 1e-5,
) -> pd.DataFrame:
    architecture = get_finalist_architecture(finalist_rank)
    run_name = fine_tune_run_name(
        architecture,
        fine_tune_layers=fine_tune_layers,
        phase2_learning_rate=phase2_learning_rate,
    )
    return inspect_model_results(run_name)


def run_screening_shortlist() -> pd.DataFrame:
    for architecture in SCREENING_ARCHITECTURES:
        run_species_phase1_screening(architecture)
    return rank_screening_results()


def add_ranking_score(
    summary: pd.DataFrame,
    metric_specs: list[tuple[str, float, bool]],
) -> pd.DataFrame:
    if summary.empty:
        return summary

    ranked = summary.copy()
    score_columns = []
    for metric, weight, higher_is_better in metric_specs:
        if metric not in ranked.columns:
            continue
        values = pd.to_numeric(ranked[metric], errors="coerce")
        if values.notna().sum() == 0:
            continue
        filled_values = values.fillna(values.min() if higher_is_better else values.max())
        score_column = f"score_{metric}"
        ranked[score_column] = filled_values.rank(
            method="average",
            pct=True,
            ascending=higher_is_better,
        ) * weight
        score_columns.append(score_column)

    if not score_columns:
        return ranked

    ranked["ranking_score"] = ranked[score_columns].sum(axis=1)
    ranked["ranking_position"] = ranked["ranking_score"].rank(
        method="first",
        ascending=False,
    ).astype(int)
    return ranked.sort_values(
        ["ranking_score", "eval_val_f1_macro"],
        ascending=[False, False],
    )


def select_diverse_ranked_runs(
    ranked: pd.DataFrame,
    *,
    top_k: int,
    max_per_family: int = 1,
) -> pd.DataFrame:
    if ranked.empty:
        return ranked

    selected_indices = []
    family_counts: dict[str, int] = {}
    for row_index, row in ranked.iterrows():
        family = row.get("architecture_family", "unknown")
        if family_counts.get(family, 0) >= max_per_family:
            continue
        selected_indices.append(row_index)
        family_counts[family] = family_counts.get(family, 0) + 1
        if len(selected_indices) == top_k:
            return ranked.loc[selected_indices]

    for row_index, row in ranked.iterrows():
        if row_index in selected_indices:
            continue
        selected_indices.append(row_index)
        if len(selected_indices) == top_k:
            break
    return ranked.loc[selected_indices]


def rank_screening_results() -> pd.DataFrame:
    return add_ranking_score(load_summary(SCREENING_RESULTS_PATH), SCREENING_RANKING_METRICS)


def rank_fine_tune_results() -> pd.DataFrame:
    return add_ranking_score(load_summary(FINE_TUNE_RESULTS_PATH), FINAL_RANKING_METRICS)


def select_screening_finalists(top_k: int = 4) -> list[str]:
    ranked = rank_screening_results()
    selected = select_diverse_ranked_runs(ranked, top_k=top_k, max_per_family=2)
    return selected["architecture"].tolist() if not selected.empty else []


def select_final_ensemble(top_k: int = 3) -> pd.DataFrame:
    ranked = rank_fine_tune_results()
    return select_diverse_ranked_runs(ranked, top_k=top_k, max_per_family=1)


def plot_training_curves(run_name: str) -> None:
    history = load_history(run_name)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for phase, phase_history in history.items():
        if not phase_history:
            continue
        label = phase.replace("phase", "phase ")
        axes[0].plot(phase_history.get("loss", []), label=f"{label} train")
        axes[0].plot(phase_history.get("val_loss", []), label=f"{label} val")
        axes[1].plot(phase_history.get("accuracy", []), label=f"{label} train")
        axes[1].plot(phase_history.get("val_accuracy", []), label=f"{label} val")

    axes[0].set_title(f"{run_name} - loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].set_title(f"{run_name} - accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def classification_report_df(run_name: str, eval_name: str = "test") -> pd.DataFrame:
    details = evaluation_details.get((run_name, eval_name))
    if details is None:
        return pd.DataFrame()

    report = classification_report(
        details["y_true"],
        details["y_pred"],
        labels=np.arange(NUM_CLASSES),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(report).transpose()


def plot_confusion_matrix(
    run_name: str,
    eval_name: str = "test",
    normalize: bool = True,
) -> pd.DataFrame:
    details = evaluation_details.get((run_name, eval_name))
    if details is None:
        return pd.DataFrame()

    matrix = confusion_matrix(
        details["y_true"],
        details["y_pred"],
        labels=np.arange(NUM_CLASSES),
        normalize="true" if normalize else None,
    )
    matrix_df = pd.DataFrame(matrix, index=class_names, columns=class_names)

    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(matrix_df.values, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    ax.set_title(f"{run_name} - matrice de confusion {eval_name}")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Vrai label")
    ax.set_xticks(np.arange(NUM_CLASSES), labels=class_names, rotation=45, ha="right")
    ax.set_yticks(np.arange(NUM_CLASSES), labels=class_names)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    value_format = ".2f" if normalize else ".0f"
    for row_index in range(NUM_CLASSES):
        for column_index in range(NUM_CLASSES):
            value = matrix_df.iloc[row_index, column_index]
            ax.text(
                column_index,
                row_index,
                format(value, value_format),
                ha="center",
                va="center",
                color="white" if value > 0.5 else "black",
                fontsize=8,
            )

    plt.tight_layout()
    plt.show()
    return matrix_df


def inspect_model_results(
    run_name: str,
    *,
    results_path: Path = FINE_TUNE_RESULTS_PATH,
    eval_name: str = "test",
) -> pd.DataFrame:
    plot_training_curves(run_name)
    summary = load_summary(results_path)
    model_summary = summary[summary["run_name"] == run_name] if not summary.empty else summary
    display(model_summary)
    if not model_summary.empty and "overfit_diagnostic" in model_summary.columns:
        diagnostic = model_summary["overfit_diagnostic"].iloc[0]
        accuracy_gap = model_summary.get("overfit_gap", pd.Series([None])).iloc[0]
        validation_loss_gap = model_summary.get("loss_gap", pd.Series([None])).iloc[0]
        print(
            f"Diagnostic overfitting: {diagnostic} "
            f"(gap accuracy train-val={accuracy_gap:.4f}, "
            f"gap loss val-train={validation_loss_gap:.4f})"
        )

    report = classification_report_df(run_name, eval_name=eval_name)
    if report.empty:
        print(
            "Rapport par classe et matrice de confusion disponibles apres avoir "
            "evalue ce run dans la session notebook courante."
        )
    else:
        display(report)
        plot_confusion_matrix(run_name, eval_name=eval_name)

    return model_summary

SCREENING_ARCHITECTURES, DEFERRED_ARCHITECTURES, rank_screening_results()


(['MobileNetV3Small',
  'MobileNetV3Large',
  'EfficientNetB0',
  'EfficientNetB1',
  'ConvNeXtTiny',
  'DenseNet121'],
 ['EfficientNetB3',
  'ResNet50V2',
  'ResNet101V2',
  'ConvNeXtSmall',
  'DenseNet169'],
                      run_name      architecture architecture_family  \
 0    EfficientNetB0_screening    EfficientNetB0        EfficientNet   
 2  MobileNetV3Large_screening  MobileNetV3Large           MobileNet   
 1      ConvNeXtTiny_screening      ConvNeXtTiny            ConvNeXt   
 3    EfficientNetB1_screening    EfficientNetB1        EfficientNet   
 4       DenseNet121_screening       DenseNet121            DenseNet   
 5  MobileNetV3Small_screening  MobileNetV3Small           MobileNet   
 
               stage  phase1_learning_rate  phase2_learning_rate  \
 0  screening_phase1                 0.001                   NaN   
 2  screening_phase1                 0.001                   NaN   
 1  screening_phase1                 0.001                   NaN   
 3  screenin

## Phase 1 - Screening rapide

On entraine uniquement la tete de classification avec le backbone gele. Cette etape sert a filtrer vite plusieurs familles d'architectures sans payer le cout du fine-tuning complet.

Le ranking de screening combine F1 macro validation, balanced accuracy, log loss, temps d'inference et signaux d'overfitting. Le test set reste reserve aux finalistes fine-tunes.

In [ ]:
# Option bulk si tu veux lancer toute la shortlist d'un coup.
# Sinon, laisse cette ligne commentee et utilise les cellules manuelles une architecture a la fois.
# screening_summary = run_screening_shortlist()

rank_screening_results()

In [ ]:
finalist_architectures = select_screening_finalists(top_k=4)
finalist_architectures

## Option - Screening manuel

Si tu ne veux pas lancer toute la shortlist, execute uniquement les cellules ci-dessous, une architecture a la fois.

### Screening - MobileNetV3Small

In [ ]:
mobilenet__v3_small_screening = run_species_phase1_screening(
    "MobileNetV3Small",
    phase1_epochs=4,
)
mobilenet__v3_small_screening

In [ ]:
inspect_model_results(
    "MobileNetV3Small_screening",
    results_path=SCREENING_RESULTS_PATH,
    eval_name="val",
)

### Screening - MobileNetV3Large

In [ ]:
mobilenet__v3_large_screening = run_species_phase1_screening(
    "MobileNetV3Large",
    phase1_epochs=4,
)
mobilenet__v3_large_screening

In [ ]:
inspect_model_results(
    "MobileNetV3Large_screening",
    results_path=SCREENING_RESULTS_PATH,
    eval_name="val",
)

### Screening - EfficientNetB0

In [ ]:
efficientnet_b0_screening = run_species_phase1_screening(
    "EfficientNetB0",
    phase1_epochs=4,
)
efficientnet_b0_screening

In [ ]:
inspect_model_results(
    "EfficientNetB0_screening",
    results_path=SCREENING_RESULTS_PATH,
    eval_name="val",
)

### Screening - EfficientNetB1

In [ ]:
efficientnet_b1_screening = run_species_phase1_screening(
    "EfficientNetB1",
    phase1_epochs=4,
)
efficientnet_b1_screening

In [ ]:
inspect_model_results(
    "EfficientNetB1_screening",
    results_path=SCREENING_RESULTS_PATH,
    eval_name="val",
)

### Screening - ConvNeXtTiny

In [ ]:
convnext_tiny_screening = run_species_phase1_screening(
    "ConvNeXtTiny",
    phase1_epochs=4,
)
convnext_tiny_screening

In [ ]:
inspect_model_results(
    "ConvNeXtTiny_screening",
    results_path=SCREENING_RESULTS_PATH,
    eval_name="val",
)

### Screening - DenseNet121

In [ ]:
densenet_121_screening = run_species_phase1_screening(
    "DenseNet121",
    phase1_epochs=4,
)
densenet_121_screening

In [ ]:
inspect_model_results(
    "DenseNet121_screening",
    results_path=SCREENING_RESULTS_PATH,
    eval_name="val",
)

## Phase 2 - Fine-tuning des finalistes

On garde plus de 3 finalistes apres screening, par defaut 4, pour que la selection finale ne soit pas automatique par construction.

Le fine-tuning evalue ensuite val, test in-distribution et OOD PlantDoc. Le ranking final combine les metriques test, OOD, temps d'inference et overfitting, puis selectionne 3 modeles en favorisant la diversite de familles pour le vote doux.

In [ ]:
# A ajuster apres lecture du screening_summary.
# Exemple: ["MobileNetV3Large", "EfficientNetB0", "DenseNet121"]
finalist_architectures = select_screening_finalists(top_k=4)
finalist_architectures

In [ ]:
# Option bulk si tu veux fine-tuner tous les finalistes d'un coup.
# Sinon, utilise les cellules manuelles ci-dessous.
# fine_tune_results = []
# for architecture in finalist_architectures:
#     fine_tune_results.append(
#         run_species_fine_tuning(
#             architecture,
#             fine_tune_layers=50,
#             phase2_learning_rate=1e-5,
#             phase2_epochs=10,
#         )
#     )

rank_fine_tune_results()

## Option - Fine-tuning manuel

Ces cellules suivent automatiquement `finalist_architectures`. Si tu veux forcer une selection, modifie la cellule juste au-dessus avec une liste comme `finalist_architectures = ["EfficientNetB0", "ConvNeXtTiny", "MobileNetV3Large", "DenseNet121"]`.


### Fine-tuning - Finaliste 1


In [ ]:
finalist_1_ft_l50 = run_finalist_fine_tuning(
    1,
    fine_tune_layers=50,
    phase2_learning_rate=1e-5,
    phase2_epochs=10,
)
finalist_1_ft_l50


In [ ]:
inspect_finalist_fine_tuning(1)


### Fine-tuning - Finaliste 2


In [ ]:
finalist_2_ft_l50 = run_finalist_fine_tuning(
    2,
    fine_tune_layers=50,
    phase2_learning_rate=1e-5,
    phase2_epochs=10,
)
finalist_2_ft_l50


In [ ]:
inspect_finalist_fine_tuning(2)


### Fine-tuning - Finaliste 3


In [ ]:
finalist_3_ft_l50 = run_finalist_fine_tuning(
    3,
    fine_tune_layers=50,
    phase2_learning_rate=1e-5,
    phase2_epochs=10,
)
finalist_3_ft_l50


In [ ]:
inspect_finalist_fine_tuning(3)


### Fine-tuning - Finaliste 4


In [ ]:
finalist_4_ft_l50 = run_finalist_fine_tuning(
    4,
    fine_tune_layers=50,
    phase2_learning_rate=1e-5,
    phase2_epochs=10,
)
finalist_4_ft_l50


In [ ]:
inspect_finalist_fine_tuning(4)


## Synthese

Le choix final retient 3 modeles pour l'ensemble par vote doux. La selection automatique sert de base objective; elle doit rester validable par lecture des matrices de confusion, de l'OOD PlantDoc et de l'UI DagsHub/MLflow.

In [30]:
screening_ranking = rank_screening_results()
finetune_ranking = rank_fine_tune_results()
final_ensemble = select_final_ensemble(top_k=3)
screening_ranking, finetune_ranking, final_ensemble

(                     run_name      architecture architecture_family  \
 0    EfficientNetB0_screening    EfficientNetB0        EfficientNet   
 2  MobileNetV3Large_screening  MobileNetV3Large           MobileNet   
 1      ConvNeXtTiny_screening      ConvNeXtTiny            ConvNeXt   
 3    EfficientNetB1_screening    EfficientNetB1        EfficientNet   
 4       DenseNet121_screening       DenseNet121            DenseNet   
 5  MobileNetV3Small_screening  MobileNetV3Small           MobileNet   
 
               stage  phase1_learning_rate  phase2_learning_rate  \
 0  screening_phase1                 0.001                   NaN   
 2  screening_phase1                 0.001                   NaN   
 1  screening_phase1                 0.001                   NaN   
 3  screening_phase1                 0.001                   NaN   
 4  screening_phase1                 0.001                   NaN   
 5  screening_phase1                 0.001                   NaN   
 
    fine_tuned_l

In [31]:
final_ensemble = select_final_ensemble(top_k=3)
selected_species_models = final_ensemble["run_name"].tolist() if not final_ensemble.empty else []
selection_notes = (
    "Selection automatique par score multi-metriques avec diversite de familles. "
    "A valider visuellement dans DagsHub et avec les matrices de confusion."
)
selected_species_models, selection_notes

(['ConvNeXtTiny_ft_l50_lr1e_05',
  'EfficientNetB0_ft_l50_lr1e_05',
  'MobileNetV3Large_ft_l50_lr1e_05'],
 'Selection automatique par score multi-metriques avec diversite de familles. A valider visuellement dans DagsHub et avec les matrices de confusion.')